# Step 2: Filter notebook

Ce notebook filtre les attributs et uniformise le format afin qu'ils soient prêt à être intégré à l'indice. Les filtres sont évalués au cas par cas selon la pertinence.

In [ ]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
import rasterio
import numpy as np
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from feature_to_network import *

## Imports

#### Import des segments

In [ ]:
# operation_crs = "EPSG:2056"  # Swiss coordinate system
# target_crs = "EPSG:4326"  # WGS84 coordinate system

# print("Set parameters : GG or GE, bike or walk")
# territory = 'GG' # GG or GE or GGminGE
# network = "walk"  # walk or bike

# if territory == 'GG':
#     input_file_path = '../../Data/input'
#     output_step1_path='../../Data/output/GG/step-1'
#     output_step2_path='../../Data/output/GG/step-2'
#     output_step3_path='../../Data/output/GG/step-3'


#     save_path = '../../Data/output/GG/step-2'

#     attribute_source_path = 'source_path_GG'
#     file_name = 'file_name_GG'  # column name in attributs_info excel

# if territory == 'GE':
#     input_file_path = '../../Data/input'
#     output_step1_path='../../Data/output/GE/step-1'
#     output_step2_path='../../Data/output/GE/step-2'
#     output_step3_path='../../Data/output/GE/step-3'


#     save_path = '../../Data/output/GE/step-2'

#     attribute_source_path = 'source_path_GE'
#     file_name = 'file_name_GE'  # column name in attributs_info excel


# save_filtered_attributes = True

# # Load segments GeoDataFrame (with 'segment_id')
# print("Loading segments...")
# # reLoad pedestrian segments
# segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
# segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
# segmented_net = segmented_net.to_crs(operation_crs)





operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "walk"  # walk ONLY in the notebook, go to 2_1_Filter_features_bike.ipynb for bike

input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-2"
attribute_source_path = f"source_path_{territory}"
file_name = f"file_name_{territory}"  # column name in attributs_info excel

save_filtered_attributes = True
include = f"include_in_index_{territory}"


# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad walk segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


In [ ]:
import geopandas as gpd
from shapely.geometry.base import BaseGeometry

# Clean geometries safely
def clean_any_geom(geom):
    if geom is None:
        return None
    if not isinstance(geom, BaseGeometry):
        return None
    if geom.is_valid:
        return geom
    try:
        fixed = geom.buffer(0)
        if fixed is None or fixed.is_empty:
            return geom
        return fixed
    except Exception:
        return geom
        

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

#### Import des attributs 

In [ ]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}.xlsx", sheet_name="attributs_info")

# Keep the full attribute table available for per-attribute cells.
# Excel may read include_in_index_* either as booleans or strings depending on the column.
attributs_info["include_in_current_index"] = (
    attributs_info[include]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)
attributs_info["attribute_available_for_territory"] = ~(
    attributs_info[attribute_source_path]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["nan", "none", "", "unavailable"])
)

print(f"Attributes included in index for {territory}: {attributs_info['include_in_current_index'].sum()} / {len(attributs_info)}")
print(f"Attributes available for {territory}: {attributs_info['attribute_available_for_territory'].sum()} / {len(attributs_info)}")

attributs_info

In [ ]:
import fiona

gpkg_path = f"{input_file_path}/attributs/GG/osm/osm_attributes.gpkg"
print(fiona.listlayers(gpkg_path))

**Attribut Accidents**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----

attribute = 'accident'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
gdf = gpd.read_file(
    f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}",
    encoding="latin1"   # or "ISO-8859-1"
)
gdf = gdf.to_crs(target_crs)

gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
gdf = gdf[gdf.geometry.notna()].copy()
print(f"Processing attribute: {attribute}")

# Filter
# Initialize
gdf['filtered'] = 0

# Filter criterion
###----change here----------------------

gdf['filtered'] = 1

###--------------------------------------
print("Filters applied")

# Count and proportion
print(f"Proportion of features {attribute} kept after filtering:")
print(gdf.filtered.value_counts(normalize=True))

# Save
save(save_filtered_attributes, row, gdf, attribute)



# 
# # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'accident'
# ###--------------------

# if territory == 'GE':

#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    
#     # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")
#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)

#         gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
#         gdf = gdf[gdf.geometry.notna()].copy()
#         print(f"Processing attribute: {attribute}")

#         # Filter
#         gdf['filtered'] = 0

#         ###----change here----------------------
#         gdf.loc[gdf.ANNEE > 2020, 'filtered'] = 1
#         gdf.loc[gdf.CONSEQ.isin(['Avec blessés graves', 'Avec tués']), 'filtered'] = 1
#         gdf.loc[gdf.VELOS == 1, 'filtered'] = 1
#         gdf.loc[gdf.VAE_25 == 1, 'filtered'] = 1
#         gdf.loc[gdf.VAE_45 == 1, 'filtered'] = 1
#         gdf.loc[gdf.PIETONS == 1, 'filtered'] = 1
#         ###--------------------------------------
#         print("Filters applied")

#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         save(save_filtered_attributes, row, gdf, attribute)

# if territory == 'GG':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    
#     # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")
#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)

#         gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
#         gdf = gdf[gdf.geometry.notna()].copy()
#         print(f"Processing attribute: {attribute}")

#         # Filter
#         gdf['filtered'] = 0

#         ###----change here----------------------
#         gdf['filtered'] = 1

#         ###--------------------------------------
#         print("Filters applied")

#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         save(save_filtered_attributes, row, gdf, attribute)

**Attribut Vitesse**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'vitesse'
###--------------------

if territory == 'GE':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:   
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_ZONE == 0, 'filtered'] = 1


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
        
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        
        gdf['filtered'] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)


**Attribut Zone pietonne**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_pietonne'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_ZONE.isin(['Zone piétonne', 'Zone de rencontre (20)', ]), 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1



        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Zone apaisée** 

In [ ]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'zone_apaisee'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE_LIMIT.isin(['Zone 30 km/h', 'Prescription 30 km/h', 'Prescription 20 km/h']), 'filtered'] = 1 

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:

        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[(gdf.speed_kph > 5) & (gdf.speed_kph < 30), 'filtered'] = 1



        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Lac et cours d'eau**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'lac_cours_deau'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.GENRE.isin(["eau stagnante", "cours d'eau"]), 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)


**Fontaines**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'fontaine'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Rez Actifs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'rez_actif'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0


        # Filter criterion
        ###----change here----------------------
        keywords = ['commerce de détail', 'droguerie', 'Hôtels', 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Restaurants', 'Salons', 'Écoles']

        rez_actif_keywords = [
            # Commerce de détail (générique)
            "commerce de détail",  # capte toutes les branches "commerce de détail ..."
            "grands magasins",
            "hypermarchés",
            "grands supermarchés", "petits supermarchés",
            "grands commerces", "petits commerces",
            "autres commerces de détail en magasin spécialisé",
            "autres commerces de détail en magasin non spécialisé",
            # "autres commerces de détail hors magasin, éventaires ou marchés",
            "commerce de détail alimentaire sur éventaires et marchés",

            # Hotel-restauration
            "restaurants, cafés, snack-bar",  # ligne exacte dans ta liste
            "restaurants avec possibilité d'hébergement",
            "services des traiteurs",
            "autres services de restauration",
            "bars",
            "boulangeries – tea-rooms",
            "salons de dégustation",
            "hôtels, auberges et pensions avec restaurant",
            "hôtels, auberges et pensions sans restaurant",
            # "hébergement collectif",
            # "autres hébergements",

            # Services de proximité et bien-être
            "salons de coiffure",
            "instituts de beauté",
            "parfumeries",
            "autres activités visant au bien-être physique",
            "centres de gymnastique et de fitness",
            "studios photographiques",
            "blanchisserie",
            "nettoyage à sec",
            "réparation de chaussures",
            "réparation d'autres biens personnels et domestiques",
            "réparation d'articles d'horlogerie et de bijouterie",
            "réparation de meubles et d'équipements du foyer",

            # Banques et assurances avec guichet
            "banques cantonales",
            "banques commerciales",
            "banques raiffeisen",
            "grandes banques",
            "autres banques",
            "succursales de banques étrangères",
            # "banquiers privés",
            # "établissements spécialisés dans le prêt personnel",
            # "caisses maladie",
            # "caisses de compensation",
            # "assurances contre les accidents et les dommages",
            # "assurance vie",
            # "autres assurances (sans la sécurité sociale obligatoire)",

            # Agences en contact avec le public
            "agences immobilières",
            "activités des marchands de biens immobiliers",
            "promotion immobilière",
            "activités des agences de voyage",
            "voyagistes; tour operator",
            "agences de placement de main-d'oeuvre",
            "agences de travail temporaire",
            # je n’inclus plus : centres d'appels, portails internet, agences de recouvrement (souvent en étage / back-office)

            # Culture / loisirs à rez de chaussée
            "gestion de salles de spectacles",
            "salles de spectacles",  # si présent
            "projection de films cinématographiques; cinémas",
            "cinémas",
            "discothèques, dancings, night clubs",
            "discothèques", "night clubs",
            "organisation de jeux de hasard et d'argent",

            # Funéraire
            "services funéraires",
        ]

        def flag_with_keywords(series, keywords):
            s = series.fillna("").str.lower()
            return s.apply(lambda v: any(kw.lower() in v for kw in keywords))

        gdf["filtered"] = flag_with_keywords(gdf["BRANCHE"], rez_actif_keywords).astype(int)


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")
        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        rez_actif_set = {
        "restaurant","cafe","bar","pub","fast_food","ice_cream","biergarten",
        "bank","atm","bureau_de_change",
        "cinema","theatre","nightclub","casino",
        "marketplace","vending_machine"
        }

        gdf.loc[gdf["amenity"].isin(rez_actif_set), "filtered"] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)



**Attribut Bruit**

In [ ]:
attribute_source_path

In [ ]:

# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'bruit'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # # Simplify geometries
        # simplify_tolerance = 0.01  # in meters
        # print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        # gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        # print("Simplification done.")
        # ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # # Simplify geometries
        # simplify_tolerance = 0.01  # in meters
        # print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        # gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        # print("Simplification done.")
        # ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Proximité TP**



In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'tp'
###--------------------

if territory == 'GE':

    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Stationnement Genant**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'stationnement_genant'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
        gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")


        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf[gdf.geometry.notna()].copy()  # Remove rows with None geometries
        gdf = gdf[gdf.geometry.is_valid].copy()  # Keep only valid geometries
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")


        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Proximité Aménités**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'amenite'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        #keep if the column "BRANCHE" contains one of the following keywords (case insensitive)
        keywords = ['commerce de détail', 'droguerie', 'enseignement', 'Hôpitaux', 'Hôtels', "Maisons", 'Paroisses et associations religieuses', 'Petits commerces', 'Petits supermarchés', 'Protection civile', 'Restaurants', 'Salons', 'Écoles'  ]
        amenity_keywords = [
            # Santé et social
            "hôpital", "hôpitaux", "soins généraux",
            "hôpitaux spécialisés",
            "médecin", "médecins", "médecins généralistes",
            "dentair",  # dentaire, dentaires
            "laboratoires médicaux", "laboratoire médical",
            "paramédical", "paramédicales",
            "psychothérap", "psychologie",
            #"physiothérapie", "physio",
            #"infirmières", "soins à domicile",
            #"organisations pour la santé",
            #"institutions pour personnes handicapées",
            #"maisons pour personnes âgées",
            #"foyers pour enfants", "foyers pour enfants et adolescents",
            #"établissements pour les traitements psychosociaux",
            #"toxicomanes",
            #"action sociale",  # avec ou sans hébergement

            # Éducation
            "enseignement pré-primaire",
            "écoles primaires",
            "écoles obligatoires",
            "écoles de degré secondaire",
            "écoles de degré secondaire ii",
            "enseignement post-secondaire",
            "formation professionnelle de base",
            "formation professionnelle supérieure",
            "formation professionnelle",
            #"cours de langues",
            "enseignement culturel",
            "autres activités d'enseignements",
            "hautes écoles universitaires",
            "hautes écoles spécialisées",
            "hautes écoles pédagogiques",
            "écoles à programmes d'enseignement spécial",
            "cours de perfectionnement professionnel",

            # Culture, loisirs publics
            "gestion des bibliothèques", "bibliothèques", "archives",
            "gestion des musées",
            "gestion des sites et monuments historiques",
            "attractions touristiques similaires",
            "projection de films cinématographiques; cinémas",
            "cinémas",  # certaines lignes écrites différemment
            "orchestres, choeurs, musiciens",
            "troupes de théâtre et de ballet",
            "autres activités récréatives et de loisirs",
            "parcs d'attractions", "parcs à thèmes",
            "jardins botaniques", "jardins zoologiques", "réserves naturelles",

            # Sport
            "associations sportives", "piscine", "centre aquatique", "natation"
            "gestion d'installations sportives",
            "activités liées au sport",
            "enseignement de disciplines sportives",
            # (les centres de fitness peuvent être aussi rez actifs, on les met dans les deux)

            # Culte
            "paroisses et associations religieuses",
            "couvents et congrégations",

            # Services publics et sécurité
            #"administration publique générale",
            #"administration publique (tutelle)",
            #"poste dans le cadre d'une obligation de service universel",
            #"services du feu et de secours",
            #"activités d'ordre public et de sécurité",
            #"défense",
            "protection civile",
            "ambassades",
            "consulats",
            #"assurance-vieillesse et survivants",  # AVS, AI, AC
            #"assurance-invalidité",
            #"assurance-chômage",

            # Social élargi
            #"maisons d'éducation",
            #"foyers pour enfants et adolescents",
            #"appartements, maisons de vacances",  # si tu veux compter comme aménités touristiques
        ]



        def flag_with_keywords(series, keywords):
            s = series.fillna("").str.lower()
            return s.apply(lambda v: any(kw.lower() in v for kw in keywords))

        gdf["filtered"] = flag_with_keywords(gdf["BRANCHE"], amenity_keywords).astype(int)


        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")
        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        amenite_set = {
        "hospital","clinic","doctors","dentist","pharmacy",
        "school","kindergarten","college","university",
        "library","public_bookcase","arts_centre","museum",
        "place_of_worship",
        "police","fire_station","townhall","courthouse",
        "post_office",
        "community_centre","social_facility"
        }

        gdf.loc[gdf["amenity"].isin(amenite_set), "filtered"] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut espaces ouverts**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'espaces_ouverts'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.TYPE == 'ESPACE PUBLIC', 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Confort thermique**

In [ ]:
# Initialize

attribute = 'temperature'
if territory == 'GE':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # Create GeoDataFrame directly with filtered points
            gdf = gpd.GeoDataFrame({
                'temperature': temp_values,
                'filtered': 1,  # All points are filtered since we pre-filtered the data
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            # Set CRS and convert to target CRS
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['temperature'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)



if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # Create GeoDataFrame directly with filtered points
            gdf = gpd.GeoDataFrame({
                'temperature': temp_values,
                'filtered': 1,  # All points are filtered since we pre-filtered the data
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            # Set CRS and convert to target CRS
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['temperature'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)


**Attribut Largeur trottoir**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'largeur_trottoir'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Largeur_score'] = 0
        gdf['Largeur_score'] = gdf['Largeur'].map({'Très large': 5, 'Large': 4, 'Moyen': 3, 'Etroit': 2, 'Très étroit': 1, 'Pas de trottoir': 0})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Largeur_score'] = 0
        gdf['Largeur_score'] = gdf['Largeur'].map({'Très large': 5, 'Large': 4, 'Moyen': 3, 'Etroit': 2, 'Très étroit': 1, 'Pas de trottoir': 0})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Chemin**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'chemin'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]

    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.OBJET == "Chemin" , 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf.loc[gdf.highway=='path', 'filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut conflit usages**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'conflit_md'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['Partage_us_score'] = 0
        gdf['Partage_us_score'] = gdf['Partage_us'].map({'Trafic motorisé' : 4, 'Mixité vélos - Piste sur trotto':3, 'Mixité ayants droit motorisés':2, 'Mixité vélos':1, 'Mixité vélo':1})
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        gdf['severity_score'] = 0
        gdf['severity_score'] = gdf['severity'].map({'high' : 3, 'medium':2, 'low':1})
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Topographie**

In [ ]:
## !!! Depreciated since june 2026, now computed in 1_1
# 
# # # Initialize
# attribute = None
# row = None
# gdf = None
# print("Data initialized")

# # Load
# ###----change here-----
# attribute = 'pente'
# ###--------------------

# if territory == 'GE':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
#         # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")

#     else:
#         gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
#         gdf = gdf.to_crs(target_crs)
#         print(f"Processing attribute: {attribute}")


#         # Filter
#         # Initialize
#         gdf['filtered'] = 0

#         # Filter criterion
#         ###----change here----------------------
#         gdf['filtered'] = 1
#         ###--------------------------------------
#         print("Filters applied")

#         # Count and proportion
#         print(f"Proportion of features {attribute} kept after filtering:")
#         print(gdf.filtered.value_counts(normalize=True))

#         # Save
#         save(save_filtered_attributes, row, gdf, attribute)


# if territory == 'GG':
#     row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
#         # Check availability
#     if row[attribute_source_path] == 'unavailable':
#         print(f"{attribute} non disponible pour le territoire {territory}")

#     else:
#         raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

#         with rasterio.open(raster_path) as src:
#             # Read the raster data
#             raster_data = src.read(1)  # Read first band
            
#             # Define NoData
#             nodata_val = src.nodata if src.nodata is not None else -9999
            
#             # Valid mask: remove NoData
#             valid_mask = (raster_data != nodata_val)
            
#             # Get coordinates of valid points
#             rows, cols = np.where(valid_mask)
            
#             # Sample every 10th point
#             rows, cols = rows[::10], cols[::10]
            
#             # Get coordinates in map units
#             xs, ys = rasterio.transform.xy(src.transform, rows, cols)
#             temp_values = raster_data[rows, cols]
            
#             # Create GeoDataFrame directly with filtered points
#             gdf = gpd.GeoDataFrame({
#                 'slope_degree': temp_values,
#                 'filtered': 1,  # All points are filtered since we pre-filtered the data
#                 'geometry': [Point(x, y) for x, y in zip(xs, ys)]
#             }, geometry='geometry')
            
#             # Set CRS and convert to target CRS
#             gdf = gdf.set_crs(src.crs)
#             gdf = gdf.to_crs(target_crs)

#         print(f"Processing attribute: {attribute}")
#         print("\nDescriptive statistics:")
#         print(gdf['slope_degree'].describe())

#         # Save
#         print(f"Saving {attribute}: can take up to 3mn")
# save(save_filtered_attributes, row, gdf, attribute)

**Attribut Canopée**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'canopee'
###--------------------
if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Simplify geometries
        simplify_tolerance = 0.01  # in meters
        print(f"Simplifying geometries with tolerance = {simplify_tolerance} m ...")
        gdf["geometry"] = gdf.geometry.simplify(simplify_tolerance, preserve_topology=True)
        print("Simplification done.")
        ###--------------------

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")

        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1

        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attributs Bancs**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'banc'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
        
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut Toilette**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'toilette'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")

    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut eclairage**

In [ ]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'eclairage'
###--------------------

if territory == 'GE':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

if territory == 'GG':
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        gdf = gpd.read_file(f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}", layer=row['meta_attribute'])
        gdf = gdf.to_crs(target_crs)

        gdf["geometry"] = gdf["geometry"].apply(clean_any_geom)
        gdf = gdf[gdf.geometry.notna()].copy()
        print(f"Processing attribute: {attribute}")


        # Filter
        # Initialize
        gdf['filtered'] = 0

        # Filter criterion
        ###----change here----------------------
        gdf['filtered'] = 1
        
        ###--------------------------------------
        print("Filters applied")

        # Count and proportion
        print(f"Proportion of features {attribute} kept after filtering:")
        print(gdf.filtered.value_counts(normalize=True))

        # Save
        save(save_filtered_attributes, row, gdf, attribute)

**Attribut air**

In [ ]:
# Initialize

attribute = 'air'
if territory == 'GE':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::1], cols[::1]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
            temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
            
            gdf = gpd.GeoDataFrame({
                'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
                'no_2_raw': temp_values,         # valeurs originales conservées si besoin
                'filtered': 1,
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['no_2'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)


if territory == 'GG':
    
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
        # Check availability
    if row[attribute_source_path] == 'unavailable':
        print(f"{attribute} non disponible pour le territoire {territory}")
    
    else:
        raster_path = (f"{input_file_path}/{row[attribute_source_path]}/{row[file_name]}")

        with rasterio.open(raster_path) as src:
            # Read the raster data
            raster_data = src.read(1)  # Read first band
            
            # Define NoData
            nodata_val = src.nodata if src.nodata is not None else -9999
            
            # Valid mask: remove NoData
            valid_mask = (raster_data != nodata_val)
            
            # Get coordinates of valid points
            rows, cols = np.where(valid_mask)
            
            # Sample every 10th point
            rows, cols = rows[::10], cols[::10]
            
            # Get coordinates in map units
            xs, ys = rasterio.transform.xy(src.transform, rows, cols)
            temp_values = raster_data[rows, cols]
            
            # ✅ Conversion en µmol/m² pour éviter les underflows d'affichage
            temp_values_converted = temp_values * 1e6  # mol/m² → µmol/m²
            
            gdf = gpd.GeoDataFrame({
                'no_2': temp_values_converted,   # valeurs lisibles ex: 18 à 40 µmol/m²
                'no_2_raw': temp_values,         # valeurs originales conservées si besoin
                'filtered': 1,
                'geometry': [Point(x, y) for x, y in zip(xs, ys)]
            }, geometry='geometry')
            
            gdf = gdf.set_crs(src.crs)
            gdf = gdf.to_crs(target_crs)

        print(f"Processing attribute: {attribute}")
        print("\nDescriptive statistics:")
        print(gdf['no_2'].describe())

        # Save
        print(f"Saving {attribute}: can take up to 3mn")
        save(save_filtered_attributes, row, gdf, attribute)